In [1]:
from factor_analysis_v0 import *
from pathlib import Path

data_config = DataConfig(
    underlying_gex_path=r"E:\Pythonfiles\ProjectGamma\results\option_gamma\underlying_gex_daily_with_permno.parquet",
    crsp_daily_path=r"E:\Pythonfiles\ProjectGamma\data\crsp_daily_common_2018_2024_linked_permnos.parquet",
    start_date="2018-01-01",
    end_date="2024-12-31",
    min_abs_price=1.0,
)

column_config = ColumnConfig(
    id_col="permno",
    date_col="date",
    gex_col="net_gex_1pct",
    ret_col="ret",
    spot_col="spot",
    crsp_price_col="prc",
    volume_col="vol",
    control_cols=[
        "price_stock",
        "market_equity_crsp",
        "vol",
        "shrout",
        "spot",
        "total_open_interest",
        "total_option_volume",
    ],
)

preprocess_config = PreprocessConfig(
    winsorize=True,
    winsor_lower=0.01,
    winsor_upper=0.99,
    cross_sectional_zscore=True,
    zscore_suffix="_z",
    min_cross_section_size=10,
    missing_threshold=0.95,
    allow_missing_factors=True,
    use_abs_crsp_price=True,
)

target_config = TargetConfig(
    horizons=[1, 5, 20],
    create_signed_return=True,
    create_abs_return=True,
    create_squared_return=True,
    create_realized_vol=True,
    create_downside_semivar=True,
    create_tail_indicators=True,
    tail_quantiles=[0.05, 0.95],
    tail_groupby="by_date",
    return_type="simple",
)

output_config = OutputConfig(
    output_dir=r"E:\Pythonfiles\ProjectGamma\results\gex_collab_phase1",
    save_panel_snapshot=True,
    save_metadata=True,
    panel_snapshot_name="phase1_panel.parquet",
    metadata_name="phase1_metadata.json",
    verbose=True,
)

factor_specs = [
    FactorSpec(
        name="bs_factor_csv",
        path=r"data/factors/BS.csv",
        file_type="csv",
        id_col="PERMNO",
        date_col="date",
        frequency="daily",
        factor_cols=[
            "n", "ret", "alpha", "b_mkt", "b_smb", "b_hml", "b_umd",
            "ivol", "tvol", "R2", "exret"
        ],
        numeric_only=False,
        suffix="__bs",
        lag_days=0,
    ),
]

regression_config = RegressionConfig(
    use_fama_macbeth=True,
    nw_lags=5,
    add_intercept=True,
    use_controls=True,
    control_cols=[
        "price_stock",
        "market_equity_crsp",
        "vol",
        "shrout",
        "spot",
        "total_open_interest",
        "total_option_volume",
    ],
    min_obs_per_date=10,
    min_dates_required=20,
    regime_validation_y_cols=[
        "ret_fwd_1d",
        "abs_ret_fwd_1d",
        "rv_fwd_5d",
        "tail_left_fwd_1d",
    ],
    interaction_y_cols=[
        "abs_ret_fwd_1d",
        "rv_fwd_5d",
        "tail_left_fwd_1d",
    ],
    n_buckets=5,
)

exp = GEXCollaborativeEffectExperiment(
    data_config=data_config,
    column_config=column_config,
    preprocess_config=preprocess_config,
    target_config=target_config,
    regression_config=regression_config,
    output_config=output_config,
)

In [2]:
factor_list = ["b_mkt", "b_smb", "b_hml", "b_umd", "ivol", "tvol"]
artifacts = exp.run_phase1(
    factor_specs=factor_specs,
    selected_factors=factor_list,
)

panel = artifacts["panel"]
print(panel.head())
print(panel.columns.tolist()[:100])
print(artifacts["metadata"])

    secid  permno       date   spot  spot_close  spot_return  spot_cfadj  \
0  107525   10107 2018-01-02  85.95       85.95     0.004793        16.0   
1  107525   10107 2018-01-03  86.35       86.35     0.004654        16.0   
2  107525   10107 2018-01-04  87.11       87.11     0.008801        16.0   
3  107525   10107 2018-01-05  88.19       88.19     0.012398        16.0   
4  107525   10107 2018-01-08  88.28       88.28     0.001020        16.0   

   spot_shrout  n_contracts  n_optionids  ...  rv_fwd_20d  \
0    7714590.0          726          726  ...    0.011532   
1    7714590.0          754          754  ...    0.011620   
2    7714590.0          758          758  ...    0.012875   
3    7714590.0          735          735  ...    0.015585   
4    7714590.0          752          752  ...    0.017732   

   downside_semivar_fwd_20d  tail_left_fwd_20d  tail_right_fwd_20d  \
0                  0.000019                  0                   0   
1                  0.000022         

In [3]:
artifacts_phase2 = exp.run_phase2(
    factor_cols=None,   # defaults to all phase-1 factor cols
)

E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Pythonfiles\RiskAnalysis7002\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
E:\Py